# Conditional Questions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

## Que1: Ad Campaign CTR

**Difficulty:** Easy

### Problem

An ad platform logs one row per event for every campaign, where each event is either an `impression` or a `click`.

For each campaign, calculate the click-through rate (CTR) as the number of clicks divided by the number of impressions, multiplied by 100. Round the CTR to 2 decimal places.

**Schema columns:** `ad_events.event_id`, `ad_events.campaign_id`, `ad_events.event_type`, `ad_events.event_date`

**Output columns:** `campaign_id`, `ctr`

Order the result by `campaign_id` in ascending order.

#### Example 1

**Input:**

**ad_events:**

| event_id | campaign_id | event_type | event_date |
|---------:|------------:|------------|------------|
| 1 | 101 | impression | 2024-01-01 |
| 2 | 101 | impression | 2024-01-01 |
| 3 | 101 | impression | 2024-01-01 |
| 4 | 101 | click | 2024-01-01 |
| 5 | 102 | impression | 2024-01-02 |
| 6 | 102 | impression | 2024-01-02 |
| 7 | 102 | click | 2024-01-02 |
| 8 | 102 | click | 2024-01-02 |
| 9 | 103 | impression | 2024-01-03 |

**Output:**

| campaign_id | ctr |
|------------:|----:|
| 101 | 33.33 |
| 102 | 100.00 |
| 103 | 0.00 |

**Explanation:** Campaign 101 has 3 impressions and 1 click, giving a CTR of 33.33%. Campaign 102 has 2 impressions and 2 clicks, giving a CTR of 100%. Campaign 103 has 1 impression and no clicks, giving a CTR of 0.00%.

### Constraints

- Event types are exactly `impression` or `click`.
- Every campaign has at least one impression.
- CTR is `(clicks / impressions) * 100`.
- Round CTR to 2 decimal places.
- Order the result by `campaign_id` in ascending order.

In [0]:
ad_events_data=[(1,101,"impression","2024-01-01"),(2,101,"impression","2024-01-01"),(3,101,"impression","2024-01-01"),(4,101,"click","2024-01-01"),(5,102,"impression","2024-01-02"),(6,102,"impression","2024-01-02"),(7,102,"click","2024-01-02"),(8,102,"click","2024-01-02"),(9,103,"impression","2024-01-03")]
ad_events_df=spark.createDataFrame(ad_events_data,["event_id","campaign_id","event_type","event_date"])
display(ad_events_df)

output_df = (
ad_events_df.groupBy("campaign_id").agg(
    round(
        count(when(col("event_type") == "click", 1)) / 
        count(when(col("event_type") == "impression", 2))
        * 100
    , 2).alias("ctr")
)
.orderBy("campaign_id")
)

display(output_df)



event_id,campaign_id,event_type,event_date
1,101,impression,2024-01-01
2,101,impression,2024-01-01
3,101,impression,2024-01-01
4,101,click,2024-01-01
5,102,impression,2024-01-02
6,102,impression,2024-01-02
7,102,click,2024-01-02
8,102,click,2024-01-02
9,103,impression,2024-01-03


campaign_id,ctr
101,33.33
102,100.0
103,0.0


## Que2: Immediate Food Delivery I

**Difficulty:** Easy

### Problem

A food delivery service records every order along with the date it was placed and the date the customer preferred to receive it.

An order is considered an `immediate` delivery when the order date matches the customer's preferred delivery date. Calculate the percentage of all orders that were immediate deliveries and round the result to 2 decimal places.

**Schema columns:** `delivery.delivery_id`, `delivery.customer_id`, `delivery.order_date`, `delivery.pref_delivery_date`

**Output columns:** `immediate_percentage`

### Examples

#### Example 1

**Input:**

**delivery:**

| delivery_id | customer_id | order_date | pref_delivery_date |
|------------:|------------:|------------|--------------------|
| 1 | 1 | 2019-08-01 | 2019-08-01 |
| 2 | 2 | 2019-08-02 | 2019-08-03 |
| 3 | 3 | 2019-08-03 | 2019-08-03 |
| 4 | 4 | 2019-08-04 | 2019-08-05 |
| 5 | 5 | 2019-08-05 | 2019-08-05 |

**Output:**

| immediate_percentage |
|---------------------:|
| 60.00 |

**Explanation:** Orders 1, 3, and 5 have an order date equal to the preferred delivery date. Therefore, 3 out of 5 orders are immediate deliveries, giving an immediate percentage of 60.00%.

### Constraints

- An order is immediate when `order_date` equals `pref_delivery_date`.
- Calculate the percentage over all orders.
- Round the result to 2 decimal places.
- Return a single row with `immediate_percentage`.

In [0]:
delivery_data=[(1,1,"2019-08-01","2019-08-01"),(2,2,"2019-08-02","2019-08-03"),(3,3,"2019-08-03","2019-08-03"),(4,4,"2019-08-04","2019-08-05"),(5,5,"2019-08-05","2019-08-05")]
delivery_df=spark.createDataFrame(delivery_data,["delivery_id","customer_id","order_date","pref_delivery_date"])

delivery_df = (
delivery_df
    .withColumn("is_immediate", when(col("order_date") == col("pref_delivery_date"), 1))
)

display(delivery_df)

grouped_df = (
delivery_df.agg(
    round(
        (count(col("is_immediate")) / count("delivery_id")) * 100
    , 2).alias("immediate_percentage")
))

display(grouped_df)






delivery_id,customer_id,order_date,pref_delivery_date,is_immediate
1,1,2019-08-01,2019-08-01,1
2,2,2019-08-02,2019-08-03,null
3,3,2019-08-03,2019-08-03,1
4,4,2019-08-04,2019-08-05,null
5,5,2019-08-05,2019-08-05,1


immediate_percentage
60.0


## Que3: Bank Account Summary II

**Difficulty:** Easy

### Problem

A bank maintains information about its customers' accounts and their transactions.

Calculate the balance of each account as the net amount of all its transactions, where `credit` transactions add money to the balance and `debit` transactions subtract money from the balance.

Return the `name` and `balance` of users whose balance is **higher than 10000**.

**Schema columns:** `accounts.account`, `accounts.name`, `transactions.trans_id`, `transactions.account`, `transactions.type`, `transactions.amount`

**Output columns:** `name`, `balance`

Order the result by `account` in ascending order.

### Examples

#### Example 1

**Input:**

**accounts:**

| account | name |
|--------:|------|
| 1001 | John |
| 1002 | Jane |
| 1003 | Bob |
| 1004 | Alice |
| 1005 | Charlie |

**transactions:**

| trans_id | account | type | amount |
|---------:|--------:|-------|-------:|
| 1 | 1001 | credit | 15000 |
| 2 | 1001 | debit | 2500 |
| 3 | 1002 | credit | 8000 |
| 4 | 1002 | debit | 4000 |
| 5 | 1002 | credit | 500 |
| 6 | 1003 | credit | 5000 |
| 7 | 1003 | debit | 6000 |
| 8 | 1004 | credit | 20000 |
| 9 | 1004 | debit | 3500 |
| 10 | 1004 | debit | 5500 |
| 11 | 1005 | credit | 9000 |
| 12 | 1005 | debit | 2500 |

**Output:**

| name | balance |
|------|--------:|
| John | 12500 |
| Alice | 11000 |

**Explanation:** John's balance is `15000 - 2500 = 12500`, while Alice's balance is `20000 - 3500 - 5500 = 11000`. Both balances are greater than 10000, so they are included in the result. :contentReference[oaicite:0]{index=0}

### Constraints

- `type` is either `credit` or `debit`.
- A credit increases the account balance.
- A debit decreases the account balance.
- An account may have no transactions.
- Each transaction references an existing account.
- Return only accounts whose balance is greater than `10000`.
- Order the result by `account` in ascending order.

In [0]:
accounts_data=[(1001,"John"),(1002,"Jane"),(1003,"Bob"),(1004,"Alice"),(1005,"Charlie")]
accounts_df=spark.createDataFrame(
    accounts_data,
    ["account","name"]
)
display(accounts_df)

transactions_data=[(1,1001,"credit",15000),(2,1001,"debit",2500),(3,1002,"credit",8000),(4,1002,"debit",4000),(5,1002,"credit",500),(6,1003,"credit",5000),(7,1003,"debit",6000),(8,1004,"credit",20000),(9,1004,"debit",3500),(10,1004,"debit",5500),(11,1005,"credit",9000),(12,1005,"debit",2500)]
transactions_df=spark.createDataFrame(transactions_data,["trans_id","account","type","amount"])

transactions_df = transactions_df.withColumn("amount", when(col("type") == "credit", col("amount")).otherwise(-col("amount")))

grouped_df = transactions_df.groupBy("account").agg(sum("amount").alias("balance")).filter("balance >= 10000")

joined_df = grouped_df.join(accounts_df, on="account").select("name", "balance")

display(joined_df)

account,name
1001,John
1002,Jane
1003,Bob
1004,Alice
1005,Charlie


name,balance
John,12500
Alice,11000


## Que6: One-Hot Encoding

**Difficulty:** Easy

### Problem

Convert categorical department values into binary one-hot encoded columns. Each department becomes a separate column containing `1` when the employee belongs to that department and `0` otherwise.

**Schema columns:** `employees.emp_id`, `employees.name`, `employees.department`

**Output columns:** `emp_id`, `name`, `is_engineering`, `is_hr`, `is_marketing`, `is_sales`

Sort the result by `emp_id` in ascending order.

### Examples

#### Example 1

**Input:**

**employees:**

| emp_id | name | department |
|-------:|------|------------|
| 1 | Alice Smith | Engineering |
| 2 | Bob Johnson | Marketing |
| 3 | Charlie Brown | Sales |
| 4 | Diana Prince | HR |
| 5 | Evan Davis | Engineering |

**Output:**

| emp_id | name | is_engineering | is_hr | is_marketing | is_sales |
|-------:|------|---------------:|------:|-------------:|---------:|
| 1 | Alice Smith | 1 | 0 | 0 | 0 |
| 2 | Bob Johnson | 0 | 0 | 1 | 0 |
| 3 | Charlie Brown | 0 | 0 | 0 | 1 |
| 4 | Diana Prince | 0 | 1 | 0 | 0 |
| 5 | Evan Davis | 1 | 0 | 0 | 0 |

**Explanation:** Each employee's department is converted into a separate binary column. The employee receives `1` for their department and `0` for all other departments. :contentReference[oaicite:0]{index=0}

### Constraints

- Create one binary column for each department: `Engineering`, `HR`, `Marketing`, and `Sales`.
- Use `1` for a matching department and `0` otherwise.
- Use the column order: `is_engineering`, `is_hr`, `is_marketing`, `is_sales`.
- Order the result by `emp_id` in ascending order.

In [0]:
employees_data=[(1,"Alice Smith","Engineering"),(2,"Bob Johnson","Marketing"),(3,"Charlie Brown","Sales"),(4,"Diana Prince","HR"),(5,"Evan Davis","Engineering")]
employees_df=spark.createDataFrame(employees_data,["emp_id","name","department"])
display(employees_df)

output_df = (
employees_df
    .withColumn("is_engineering", when(col("department") == "Engineering", 1).otherwise(0))
    .withColumn("is_hr", when(col("department") == "HR", 1).otherwise(0))
    .withColumn("is_marketing", when(col("department") == "Marketing", 1).otherwise(0))
    .withColumn("is_sales", when(col("department") == "Sales", 1).otherwise(0))
   .orderBy("emp_id")
)

display(output_df)

emp_id,name,department
1,Alice Smith,Engineering
2,Bob Johnson,Marketing
3,Charlie Brown,Sales
4,Diana Prince,HR
5,Evan Davis,Engineering


emp_id,name,department,is_engineering,is_hr,is_marketing,is_sales
1,Alice Smith,Engineering,1,0,0,0
2,Bob Johnson,Marketing,0,0,1,0
3,Charlie Brown,Sales,0,0,0,1
4,Diana Prince,HR,0,1,0,0
5,Evan Davis,Engineering,1,0,0,0


## Que5: Payroll Calculation Engine

**Difficulty:** Easy

### Problem

Compute weekly paychecks with overtime for the payroll run.

You are a data analyst on the payroll team at Odoo. The weekly payroll run needs each employee's gross pay computed from their logged hours, applying the standard overtime policy before checks go out.

Join `rp_employees` to `rp_payroll` on `employee_id`. If `hours_worked` is 40 or less, pay equals `hours_worked * hourly_rate`. If `hours_worked` exceeds 40, pay equals `40 * hourly_rate` plus the hours above 40 paid at `1.5` times the hourly rate. Round the pay to 2 decimal places.

**Schema columns:** `rp_employees.employee_id`, `rp_employees.name`, `rp_employees.age`, `rp_employees.position`, `rp_payroll.employee_id`, `rp_payroll.hours_worked`, `rp_payroll.hourly_rate`

**Output columns:** `employee_id`, `name`, `pay`, `position`

Order the result by `employee_id` in ascending order.

### Examples

#### Example 1

**Input:**

**rp_employees:**

| employee_id | name | age | position |
|------------:|------|----:|----------|
| 1 | Alice | 25 | Software Engineer |
| 2 | Bob | 30 | Data Analyst |
| 3 | Carol | 28 | Product Manager |
| 4 | Dave | 24 | Software Engineer |

**rp_payroll:**

| employee_id | hours_worked | hourly_rate |
|------------:|-------------:|------------:|
| 1 | 45 | 30 |
| 2 | 38 | 25 |
| 3 | 41.5 | 35 |
| 4 | 40 | 28 |

**Output:**

| employee_id | name | pay | position |
|------------:|------|----:|----------|
| 1 | Alice | 1425.00 | Software Engineer |
| 2 | Bob | 950.00 | Data Analyst |
| 3 | Carol | 1478.75 | Product Manager |
| 4 | Dave | 1120.00 | Software Engineer |

**Explanation:** Alice worked 45 hours at 30/hour. The first 40 hours give 1200, and the additional 5 hours are paid at 1.5 × 30 = 45/hour, giving a total of 1425. Bob worked 38 hours, so there is no overtime. Dave worked exactly 40 hours, which also receives no overtime premium.

### Constraints

- Overtime applies only to hours strictly above 40.
- The first 40 hours are paid at the base hourly rate.
- Overtime hours are paid at 1.5 times the hourly rate.
- Exactly 40 hours earns no overtime premium.
- Round `pay` to 2 decimal places.
- Output columns must be exactly `employee_id`, `name`, `pay`, `position`.
- Sort the result by `employee_id` in ascending order.

In [0]:
rp_employees_data=[(1,"Alice",25,"Software Engineer"),(2,"Bob",30,"Data Analyst"),(3,"Carol",28,"Product Manager"),(4,"Dave",24,"Software Engineer")]
rp_employees_df=spark.createDataFrame(rp_employees_data,["employee_id","name","age","position"])
display(rp_employees_df)

rp_payroll_data=[(1,45.0,30),(2,38.0,25),(3,41.5,35),(4,40.0,28)]
rp_payroll_df=spark.createDataFrame(rp_payroll_data,["employee_id","hours_worked","hourly_rate"])
display(rp_payroll_df)


output_df = (
    rp_employees_df
    .join(rp_payroll_df,on="employee_id",how="inner")
    .withColumn("pay",
        round(
            when(
                col("hours_worked") <= 40,
                col("hours_worked") * col("hourly_rate")
            )
            .otherwise(
                (40 * col("hourly_rate")) +
                ((col("hours_worked") - 40) * col("hourly_rate") * 1.5)
            ),
            2
        )
    )
    .select("employee_id","name","pay","position")
    .orderBy("employee_id")
)

display(output_df)

employee_id,name,age,position
1,Alice,25,Software Engineer
2,Bob,30,Data Analyst
3,Carol,28,Product Manager
4,Dave,24,Software Engineer


employee_id,hours_worked,hourly_rate
1,45.0,30
2,38.0,25
3,41.5,35
4,40.0,28


employee_id,name,pay,position
1,Alice,1425.0,Software Engineer
2,Bob,950.0,Data Analyst
3,Carol,1478.75,Product Manager
4,Dave,1120.0,Software Engineer


## Que6: Triangle Classification

**Difficulty:** Easy

### Problem

Classify each row of three side lengths. Return `Equilateral` when all sides are equal, `Isosceles` when exactly two are equal, `Scalene` when all three differ, and `Not A Triangle` when the sum of any two sides is less than or equal to the third side.

**Schema columns:** `ttc_triangle_type.a`, `ttc_triangle_type.b`, `ttc_triangle_type.c`

**Output columns:** `Classification`

### Examples

#### Example 1

**Input:**

**ttc_triangle_type:**

| a | b | c |
|---:|---:|---:|
| 20 | 20 | 23 |
| 20 | 20 | 20 |
| 20 | 21 | 22 |
| 13 | 14 | 30 |

**Output:**

| Classification |
|----------------|
| Isosceles |
| Equilateral |
| Scalene |
| Not A Triangle |

**Explanation:** The first three rows satisfy the triangle inequality with two, three, and zero equal sides. The last row is not a valid triangle because `13 + 14 <= 30`.

### Constraints

- Test triangle validity before comparing sides for equality.
- Preserve the input row order.
- Return the output column as `Classification`.

In [0]:
ttc_triangle_type_data=[(20,20,23),(20,20,20),(20,21,22),(13,14,30)]
ttc_triangle_type_df=spark.createDataFrame(ttc_triangle_type_data,["a","b","c"])
display(ttc_triangle_type_df)

output_df = (
    ttc_triangle_type_df
    .withColumn(
        "Classification",
        when(
            (col("a") + col("b") <= col("c")) |
            (col("a") + col("c") <= col("b")) |
            (col("b") + col("c") <= col("a")),
            "Not A Triangle"
        )
        .when(
            (col("a") == col("b")) &
            (col("b") == col("c")),
            "Equilateral"
        )
        .when(
            (col("a") == col("b")) |
            (col("a") == col("c")) |
            (col("b") == col("c")),
            "Isosceles"
        )
        .otherwise("Scalene")
    )
    .select("Classification")
)

display(output_df)

a,b,c
20,20,23
20,20,20
20,21,22
13,14,30


Classification
Isosceles
Equilateral
Scalene
Not A Triangle


## Que7: Conditional Column Creation

**Difficulty:** Easy

### Problem

An operations team triages incoming orders. For each order, classify its size, flag whether it ships express, and compute a priority score used to rank the queue.

The order size is `small` when the amount is below 50, `medium` when the amount is between 50 and 200 inclusive, and `large` when the amount is above 200. The express flag is 1 when the shipping type is express and 0 otherwise. The priority score equals the amount divided by 100, plus an extra 10 points when the order ships express, rounded to 2 decimals. Return every order ordered from the highest priority score to the lowest.

**Schema columns:** `orders.order_id`, `orders.customer_id`, `orders.order_date`, `orders.amount`, `orders.shipping_type`

**Output columns:** `order_id`, `customer_id`, `amount`, `shipping_type`, `order_size`, `is_express`, `priority_score`

### Examples

#### Example 1

**Input:**

**orders:**

| order_id | customer_id | order_date | amount | shipping_type |
|---------:|------------:|------------|-------:|---------------|
| 1 | 101 | 2024-01-01 | 25.50 | standard |
| 2 | 102 | 2024-01-02 | 75.00 | express |
| 5 | 105 | 2024-01-05 | 200.00 | express |
| 13 | 113 | 2024-01-13 | 300.00 | express |
| 16 | 116 | 2024-01-16 | 22.49 | standard |

**Output:**

| order_id | customer_id | amount | shipping_type | order_size | is_express | priority_score |
|---------:|------------:|-------:|---------------|-----------|:----------:|---------------:|
| 13 | 113 | 300.0 | express | large | 1 | 13.0 |
| 5 | 105 | 200.0 | express | medium | 1 | 12.0 |
| 2 | 102 | 75.0 | express | medium | 1 | 10.75 |
| 1 | 101 | 25.5 | standard | small | 0 | 0.26 |
| 16 | 116 | 22.49 | standard | small | 0 | 0.22 |

**Explanation:** Order 13 ships express, so its score is 10 + 300/100 = 13.0 and it leads the queue; order 16 is standard, so its score is just 22.49/100 = 0.22, placing it last.

### Constraints

- `order_size` is `small` when amount < 50, `medium` when 50 <= amount <= 200, and `large` when amount > 200.
- `priority_score` = amount / 100, plus 10 when shipping_type is `express`; round to 2 decimals.
- Sort by `priority_score` descending.
- Return results matching the expected output schema and order.

In [0]:
orders_data = [(1,101,"2024-01-01",25.50,"standard"),(2,102,"2024-01-02",75.00,"express"),(5,105,"2024-01-05",200.00,"express"),(13,113,"2024-01-13",300.00,"express"),(16,116,"2024-01-16",22.49,"standard")]
orders_df = spark.createDataFrame(orders_data, ["order_id","customer_id","order_date","amount","shipping_type"])
display(orders_df)

output_df = (
orders_df
    .withColumn("order_size", 
        when(col("amount") < 50, "small").
        when(col("amount") <= 200, "medium").
        otherwise("large")
    )

    .withColumn("is_express",
        when(col("shipping_type") == "express", 1).
        otherwise(0)
    
    )

    .withColumn("priorty_score", 
        when(col("is_express") == 1, round(col("amount") / 100 + 10, 2)).
        otherwise(round(col("amount") / 100, 2))
        
    )

    .orderBy(col("priorty_score").desc())
)  


display(output_df)



order_id,customer_id,order_date,amount,shipping_type
1,101,2024-01-01,25.5,standard
2,102,2024-01-02,75.0,express
5,105,2024-01-05,200.0,express
13,113,2024-01-13,300.0,express
16,116,2024-01-16,22.49,standard


order_id,customer_id,order_date,amount,shipping_type,order_size,is_express,priorty_score
13,113,2024-01-13,300.0,express,large,1,13.0
5,105,2024-01-05,200.0,express,medium,1,12.0
2,102,2024-01-02,75.0,express,medium,1,10.75
1,101,2024-01-01,25.5,standard,small,0,0.26
16,116,2024-01-16,22.49,standard,small,0,0.22


## Que8: Replace Values Conditionally

**Difficulty:** Easy

### Problem

Map survey answers to numeric values and create age group bins. Handle invalid answers by converting to NULL.

**Schema columns:** `survey_responses.response_id`, `survey_responses.question_id`, `survey_responses.answer`, `survey_responses.respondent_age`

**Output columns:** `response_id`, `question_id`, `answer_numeric`, `age_group`

Sort by the first column in ascending order.

### Examples

#### Example 1

**Input:**

**survey_responses:**

| response_id | question_id | answer | respondent_age |
|------------:|------------:|--------|---------------:|
| 1 | 101 | Yes | 25 |
| 2 | 101 | No | 45 |
| 3 | 101 | Maybe | 35 |
| 4 | 101 | N/A | 28 |
| 5 | 101 | | 55 |

**Output:**

| response_id | question_id | answer_numeric | age_group |
|------------:|------------:|---------------:|-----------|
| 1 | 101 | 1.0 | under_30 |
| 2 | 101 | 0.0 | 30_to_50 |
| 3 | 101 | 0.5 | 30_to_50 |
| 4 | 101 | NULL | under_30 |
| 5 | 101 | NULL | over_50 |

**Explanation:** The output is derived by applying the required transformations to the input data.

### Constraints

- Age groups: `under_30` (< 30), `30_to_50` (30–49), `over_50` (>= 50).
- `Yes` → 1.0, `No` → 0.0, `Maybe` → 0.5; invalid answers map to NULL.
- Order by `response_id` ASC.
- Return results matching the expected output schema and order.

In [0]:
survey_responses_data = [(1,101,"Yes",25),(2,101,"No",45),(3,101,"Maybe",35),(4,101,"N/A",28),(5,101,None,55)]
survey_responses_df = spark.createDataFrame(survey_responses_data, ["response_id","question_id","answer","respondent_age"])
display(survey_responses_df)

# response_id	question_id	answer_numeric	age_group

# Age groups: under_30 (< 30), 30_to_50 (30–49), over_50 (>= 50).
# Yes → 1.0, No → 0.0, Maybe → 0.5; invalid answers map to NULL.
# Order by response_id ASC.
# Return results matching the expected output schema and order.

output_df = (
survey_responses_df
    .withColumn("answer_numeric", 
        when(col("answer") == "Yes", 1).
        when(col("answer") == "Maybe", 0.5).
        when(col("answer") == "No", 0).
        otherwise(None)
    )

    .withColumn("age_group",
        when(col("respondent_age") < 30, "under_30").
        when(col("respondent_age") < 50, "30_to_50").
        otherwise("over_50")
    )
    .drop("answer", "respondent_age")
    .orderBy("response_id")
)

display(output_df)


response_id,question_id,answer,respondent_age
1,101,Yes,25
2,101,No,45
3,101,Maybe,35
4,101,N/A,28
5,101,null,55


response_id,question_id,answer_numeric,age_group
1,101,1.0,under_30
2,101,0.0,30_to_50
3,101,0.5,30_to_50
4,101,null,under_30
5,101,null,over_50


## Que9: Type of Triangle

**Difficulty:** Easy

### Problem

Classify each set of three side lengths. A set is `Not A Triangle` when any two sides sum to no more than the third; otherwise it is `Equilateral` when all sides match, `Isosceles` when exactly two match, and `Scalene` when all three differ. Return the original sides as `a`, `b`, and `c`, plus the classification as `triangle_type`, sorted by `a`, `b`, and `c`.

**Schema columns:** `triangles.a`, `triangles.b`, `triangles.c`

**Output columns:** `a`, `b`, `c`, `triangle_type`

### Examples

#### Example 1

**Input:**

**triangles:**

| a | b | c |
|--:|--:|--:|
| 3 | 4 | 5 |
| 3 | 3 | 3 |
| 3 | 4 | 6 |
| 5 | 5 | 7 |
| 1 | 1 | 2 |
| 10 | 10 | 10 |
| 7 | 8 | 9 |
| 5 | 12 | 13 |

**Output:**

| a | b | c | triangle_type |
|--:|--:|--:|---------------|
| 1 | 1 | 2 | Not A Triangle |
| 3 | 3 | 3 | Equilateral |
| 3 | 4 | 5 | Scalene |
| 3 | 4 | 6 | Scalene |
| 5 | 5 | 7 | Isosceles |
| 5 | 12 | 13 | Scalene |
| 7 | 8 | 9 | Scalene |
| 10 | 10 | 10 | Equilateral |

**Explanation:** For sides 1, 1, and 2, the first two sides sum to 2 rather than exceeding the third, so the row is `Not A Triangle`; 5, 5, and 7 is valid and has two equal sides, so it is `Isosceles`.

### Constraints

- Use only the records shown in the supplied tables.
- Counts and calculations follow the business rules stated above.
- Round numeric results to the precision shown in the output.
- Return results matching the expected output schema and order.

In [0]:
triangles_data = [(3,4,5),(3,3,3),(3,4,6),(5,5,7),(1,1,2),(10,10,10),(7,8,9),(5,12,13)]
triangles_df = spark.createDataFrame(triangles_data, ["a","b","c"])

display(triangles_df)

output_df = (
    triangles_df
    .withColumn("triangle_type",
        when(
            (col("a") + col("b") <= col("c")) |
            (col("a") + col("c") <= col("b")) |
            (col("b") + col("c") <= col("a")),
            "Not A Triangle"
        )
        .when(
            (col("a") == col("b")) &
            (col("b") == col("c")),
            "Equilateral"
        )
        .when(
            (col("a") == col("b")) |
            (col("a") == col("c")) |
            (col("b") == col("c")),
            "Isosceles"
        )
        .otherwise("Scalene")
    )
    .select("a", "b", "c", "triangle_type")
    .orderBy("a", "b", "c")
)

display(output_df)

a,b,c
3,4,5
3,3,3
3,4,6
5,5,7
1,1,2
10,10,10
7,8,9
5,12,13


a,b,c,triangle_type
1,1,2,Not A Triangle
3,3,3,Equilateral
3,4,5,Scalene
3,4,6,Scalene
5,5,7,Isosceles
5,12,13,Scalene
7,8,9,Scalene
10,10,10,Equilateral


## Que10: Unfinished Parts Assembly Detection

**Difficulty:** Easy

### Problem

A manufacturer tracks every component as it advances through assembly. Return components that have not received a `completion_date`; `component` identifies the part and `assembly_stage` shows its current stage. No additional sorting is required.

**Schema columns:** `ia_parts.component`, `ia_parts.completion_date`, `ia_parts.assembly_stage`

**Output columns:** `component`, `assembly_stage`

### Examples

#### Example 1

**Input:**

**ia_parts:**

| component | assembly_stage | completion_date |
|-----------|---------------:|-----------------|
| engine | 1 | 2023-03-10 |
| engine | 2 | 2023-03-11 |
| chassis | 1 | 2023-03-10 |
| chassis | 2 | |
| chassis | 3 | |
| dashboard | 1 | 2023-03-10 |
| dashboard | 2 | 2023-03-11 |
| dashboard | 3 | |

**Output:**

| component | assembly_stage |
|-----------|---------------:|
| chassis | 2 |
| chassis | 3 |
| dashboard | 3 |

**Explanation:** The chassis row at stage 2 has an empty completion date, so it appears in the output, while a row with a recorded completion date is excluded.

### Constraints

- Use only the records shown in the supplied tables.
- Counts and calculations follow the business rules stated above.
- Round numeric results to the precision shown in the output.
- Return results matching the expected output schema and order.

In [0]:
ia_parts_data = [("engine",1,"2023-03-10"),("engine",2,"2023-03-11"),("chassis",1,"2023-03-10"),("chassis",2,None),("chassis",3,None),("dashboard",1,"2023-03-10"),("dashboard",2,"2023-03-11"),("dashboard",3,None)]
ia_parts_df = spark.createDataFrame(ia_parts_data, ["component","assembly_stage","completion_date"])
display(ia_parts_df)

output_df = ia_parts_df.where(col("completion_date").isNull()).select("component", "assembly_stage")

display(output_df)

component,assembly_stage,completion_date
engine,1,2023-03-10
engine,2,2023-03-11
chassis,1,2023-03-10
chassis,2,null
chassis,3,null
dashboard,1,2023-03-10
dashboard,2,2023-03-11
dashboard,3,null


component,assembly_stage
chassis,2
chassis,3
dashboard,3


## Que11: Consecutive Order ID Swap

**Difficulty:** Medium

### Problem

Imagine a leading food delivery platform that connects customers with restaurants and cuisines, enabling them to browse menus, place orders, and receive meals at their doorstep. The platform recently encountered a glitch in its delivery instructions, causing the food items to be mismatched with the wrong orders. Each item's order was swapped with the item in the next row.

**Schema columns:** `osf_order_swap.order_id`, `osf_order_swap.item`

**Output columns:** `corrected_order_id`, `item`

Sort the results by `corrected_order_id`.

### Examples

#### Example 1

**Input:**

**osf_order_swap:**

| order_id | item |
|---------:|------|
| 1 | Chow Mein |
| 2 | Pizza |
| 3 | Pad Thai |
| 4 | Butter Chicken |

**Output:**

| corrected_order_id | item |
|-------------------:|------|
| 1 | Pizza |
| 2 | Chow Mein |
| 3 | Butter Chicken |
| 4 | Pad Thai |

### Constraints

- Handle NULL values appropriately.
- Return results matching the expected output schema and order.

In [0]:
osf_order_swap_data = [(1,"Chow Mein"),(2,"Pizza"),(3,"Pad Thai"),(4,"Butter Chicken"), (5,"Pizza")]
osf_order_swap_df = spark.createDataFrame(osf_order_swap_data, ["order_id","item"])
display(osf_order_swap_df)

# window_spec = Window.orderBy("order_id")

prev_next_df = (
osf_order_swap_df
    .withColumn("prev_item", lag("item").over(window_spec))
    .withColumn("next_item", lead("item").over(window_spec))
)

output_df = (
prev_next_df
    .withColumn("swaped_item",
        when((col("order_id") % 2 != 0) & (col("next_item").isNotNull()), col("next_item"))
        .when((col("order_id") % 2 == 0) & (col("prev_item").isNotNull()), col("prev_item"))
        .otherwise(col("item"))
    )
)


display(output_df)





order_id,item
1,Chow Mein
2,Pizza
3,Pad Thai
4,Butter Chicken
5,Pizza


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


order_id,item,prev_item,next_item,swaped_item
1,Chow Mein,null,Pizza,Pizza
2,Pizza,Chow Mein,Pad Thai,Chow Mein
3,Pad Thai,Pizza,Butter Chicken,Butter Chicken
4,Butter Chicken,Pad Thai,Pizza,Pad Thai
5,Pizza,Butter Chicken,null,Pizza


## Que12: Session Duration Bucketing

**Difficulty:** Medium

### Problem

A product team wants a distribution of session durations. Assign each session to one of five minute-based buckets and report its session count, distinct user count, average duration, and percentage of all sessions. Buckets are lower-inclusive and upper-exclusive, except the final open-ended bucket.

**Schema columns:** `sessions.session_id`, `sessions.user_id`, `sessions.start_time`, `sessions.end_time`

**Output columns:** `duration_bucket`, `session_count`, `unique_users`, `avg_duration_min`, `pct_of_total`

### Examples

#### Example 1

**Input:**

**sessions:**

| session_id | user_id | start_time | end_time |
|-----------|--------|----------------------|---------------------|
| S001 | U001 | 2024-01-01T10:00:00 | 2024-01-01T10:03:00 |
| S002 | U002 | 2024-01-01T11:00:00 | 2024-01-01T11:08:00 |
| S003 | U003 | 2024-01-01T12:00:00 | 2024-01-01T12:20:00 |
| S004 | U004 | 2024-01-01T13:00:00 | 2024-01-01T13:45:00 |
| S006 | U006 | 2024-01-02T11:00:00 | 2024-01-02T12:20:00 |

**Output:**

| duration_bucket | session_count | unique_users | avg_duration_min | pct_of_total |
|----------------|-------------:|-------------:|-----------------:|-------------:|
| 0-5 min | 1 | 1 | 3.00 | 20 |
| 5-15 min | 1 | 1 | 8.00 | 20 |
| 15-30 min | 1 | 1 | 20.00 | 20 |
| 30-60 min | 1 | 1 | 45.00 | 20 |
| 60+ min | 1 | 1 | 80.00 | 20 |

**Explanation:** The five sessions fall into different buckets, so each represents 20% of the sample.

### Constraints

- Use buckets [0,5), [5,15), [15,30), [30,60), and [60,+∞) minutes.
- Round `avg_duration_min` to 2 decimal places and `pct_of_total` to the nearest whole percentage.
- Order buckets as `0-5 min`, `5-15 min`, `15-30 min`, `30-60 min`, `60+ min`.
- Return results matching the expected output schema and order.

In [0]:
sessions_data = [("S001","U001","2024-01-01T10:00:00","2024-01-01T10:03:00"),("S002","U002","2024-01-01T11:00:00","2024-01-01T11:08:00"),("S003","U003","2024-01-01T12:00:00","2024-01-01T12:20:00"),("S004","U004","2024-01-01T13:00:00","2024-01-01T13:45:00"),("S006","U006","2024-01-02T11:00:00","2024-01-02T12:20:00")]
sessions_df = spark.createDataFrame(sessions_data, ["session_id","user_id","start_time","end_time"])
# covert times to timestamps

sessions_df = (
sessions_df
    .withColumn("start_time", to_timestamp(col("start_time")))
    .withColumn("end_time", to_timestamp(col("end_time")))
    .withColumn("duration", (unix_timestamp(col("end_time")) - unix_timestamp(col("start_time"))) / 60)
    .withColumn("duration_bucket",
        when(col("duration") < 5, "0-5 min")
        .when(col("duration") < 15, "5-15 min")
        .when(col("duration") < 30, "15-30 min")
        .when(col("duration") < 60, "30-60 min")
        .otherwise("60+ min")
    )
)

grouped_df = (
sessions_df.groupBy("duration_bucket").agg(
    count("session_id").alias("session_count"),
    countDistinct("user_id").alias("unique_users"),
    avg("duration").alias("avg_duration")
))

display(grouped_df)

duration_bucket,session_count,unique_users,avg_duration
0-5 min,1,1,3.0
5-15 min,1,1,8.0
15-30 min,1,1,20.0
30-60 min,1,1,45.0
60+ min,1,1,80.0


## Que13: Monthly Transactions I

**Difficulty:** Medium

### Problem

For each month, report a per-country summary of the transactions that occurred in that month. A transaction is considered approved when its state equals `approved`.

Return one row per month and country with these columns:
- `month`: the transaction's year and month, formatted as `YYYY-MM`.
- `country`: the country of the transactions.
- `trans_count`: the total number of transactions for that month and country.
- `approved_count`: how many of those transactions are approved.
- `trans_total_amount`: the sum of amount across all those transactions.
- `approved_total_amount`: the sum of amount across only the approved transactions.

Order the result by month, then country.

**Schema columns:** `transactions.id`, `transactions.country`, `transactions.state`, `transactions.amount`, `transactions.trans_date`

**Output columns:** `month`, `country`, `trans_count`, `approved_count`, `trans_total_amount`, `approved_total_amount`

### Examples

#### Example 1

**Input:**

**transactions:**

| id | country | state | amount | trans_date |
|---:|---------|----------|-------:|------------|
| 1 | US | approved | 100 | 2024-01-05 |
| 2 | US | rejected | 150 | 2024-01-10 |
| 3 | US | approved | 120 | 2024-01-15 |
| 4 | CA | approved | 200 | 2024-01-20 |
| 5 | CA | pending | 250 | 2024-01-25 |
| 6 | US | approved | 110 | 2024-02-03 |
| 7 | US | rejected | 130 | 2024-02-08 |

**Output:**

| month | country | trans_count | approved_count | trans_total_amount | approved_total_amount |
|-------|---------|------------:|---------------:|-------------------:|----------------------:|
| 2024-01 | CA | 2 | 1 | 450 | 200 |
| 2024-01 | US | 3 | 2 | 370 | 220 |
| 2024-02 | US | 2 | 1 | 240 | 110 |

**Explanation:** In 2024-01, the US had 3 transactions totaling 370, of which 2 are approved totaling 220. CA had 2 transactions totaling 450, but only id 4 is approved, so `approved_count` is 1 and `approved_total_amount` is 200.

### Constraints

- A transaction is approved only when `state = 'approved'`; any other state counts toward `trans_count` and `trans_total_amount` but not the approved totals.
- Group transactions by their `YYYY-MM` month and country.
- Order the result by month, then country.
- Return results matching the expected output schema and order.

In [0]:
transactions_data = [(1,"US","approved",100,"2024-01-05"),(2,"US","rejected",150,"2024-01-10"),(3,"US","approved",120,"2024-01-15"),(4,"CA","approved",200,"2024-01-20"),(5,"CA","pending",250,"2024-01-25"),(6,"US","approved",110,"2024-02-03"),(7,"US","rejected",130,"2024-02-08")]
transactions_df = spark.createDataFrame(transactions_data, ["id","country","state","amount","trans_date"])

transactions_df = (
transactions_df.withColumn("month", date_format(col("trans_date").cast("date"),"yyyy-MM"))
)

# month	country	trans_count	approved_count	trans_total_amount	approved_total_amount


display(transactions_df)
transactions_data = [(1,"US","approved",100,"2024-01-05"),(2,"US","rejected",150,"2024-01-10"),(3,"US","approved",120,"2024-01-15"),(4,"CA","approved",200,"2024-01-20"),(5,"CA","pending",250,"2024-01-25"),(6,"US","approved",110,"2024-02-03"),(7,"US","rejected",130,"2024-02-08")]
transactions_df = spark.createDataFrame(transactions_data, ["id","country","state","amount","trans_date"])

transactions_df = (
transactions_df.withColumn("month", date_format(col("trans_date").cast("date"),"yyyy-MM"))
)

# month	country	trans_count	approved_count	trans_total_amount	approved_total_amount

grouped_df = (
transactions_df.groupBy("month", "country").agg(
    count(col("id")).alias("trans_count"),
    count(when(col("state") == "approved", 1)).alias("approved_count"),
    sum(col("amount")).alias("trans_total_amount"),
    sum(when(col("state") == "approved", col("amount"))).alias("approved_total_amount"),
)
.orderBy("month", "country")
)

display(grouped_df)




id,country,state,amount,trans_date,month
1,US,approved,100,2024-01-05,2024-01
2,US,rejected,150,2024-01-10,2024-01
3,US,approved,120,2024-01-15,2024-01
4,CA,approved,200,2024-01-20,2024-01
5,CA,pending,250,2024-01-25,2024-01
6,US,approved,110,2024-02-03,2024-02
7,US,rejected,130,2024-02-08,2024-02


month,country,trans_count,approved_count,trans_total_amount,approved_total_amount
2024-01,CA,2,1,450,200
2024-01,US,3,2,370,220
2024-02,US,2,1,240,110


## Que14: Tree Node

**Difficulty:** Medium

### Problem

A table stores a binary tree as node identifiers and parent identifiers. Classify each node as `Root` when it has no parent, `Inner` when it has both a parent and at least one child, or `Leaf` otherwise; return `id` and classification as `type`, ordered by id.

**Schema columns:** `tree.id`, `tree.p_id`

**Output columns:** `id`, `type`

Order the result by `id`.

### Examples

#### Example 1

**Input:**

**tree:**

| id | p_id |
|---:|-----:|
| 1 | NULL |
| 2 | 1 |
| 3 | 1 |
| 4 | 2 |
| 5 | 2 |

**Output:**

| id | type |
|---:|------|
| 1 | Root |
| 2 | Inner |
| 3 | Leaf |
| 4 | Leaf |
| 5 | Leaf |

**Explanation:** Node 1 has no parent, so it is `Root`. Node 2 is a child of 1 and a parent of nodes 4 and 5, so it is `Inner`; node 3 has no children and is `Leaf`.

### Constraints

- A NULL `p_id` identifies a root node.
- Every non-NULL parent identifier refers to another node.
- Return results matching the expected output schema and order.

In [0]:
tree_data = [(1,None),(2,1),(3,1),(4,2),(5,2)]
tree_df = spark.createDataFrame(tree_data, ["id","p_id"])
display(tree_df)

child_count_df = (
tree_df.groupBy("p_id").agg(
    count("id").alias("child_count")
))

joined_df = tree_df.alias("t").join(child_count_df.alias("c"), on=col("t.id") == col("c.p_id"), how="left").select("t.id", "t.p_id", "c.child_count")

output_df = (
joined_df
    .withColumn("type", 
        when(col("p_id").isNull(), "Root")
        .when(col("child_count").isNotNull(), "Inner")
        .otherwise("Leaf")
    )
)


display(output_df)


id,p_id
1,null
2,1
3,1
4,2
5,2


id,p_id,child_count,type
1,null,2,Root
2,1,2,Inner
3,1,null,Leaf
4,2,null,Leaf
5,2,null,Leaf


## Que15: Capital Gain Loss

**Difficulty:** Medium

### Problem

An investment tracker calculates each stock's net capital gain or loss by subtracting Buy prices and adding Sell prices. Return `stock_name` and its `capital_gain_loss`, ordered by stock name.

**Schema columns:** `stocks.stock_name`, `stocks.operation`, `stocks.operation_day`, `stocks.price`

**Output columns:** `stock_name`, `capital_gain_loss`

Order the result by `stock_name`.

### Examples

#### Example 1

**Input:**

**stocks:**

| stock_name | operation | operation_day | price |
|------------|-----------|-------------:|------:|
| Alpha | Buy | 1 | 100 |
| Alpha | Sell | 5 | 140 |
| Beta | Buy | 2 | 80 |
| Beta | Sell | 6 | 60 |

**Output:**

| stock_name | capital_gain_loss |
|------------|------------------:|
| Alpha | 40 |
| Beta | -20 |

**Explanation:** Alpha has 140 - 100 = 40 in capital gain. Beta has 60 - 80 = -20, representing a loss.

### Constraints

- `operation` is either `Buy` or `Sell`.
- All recorded prices for a stock contribute to its net result.
- Return results matching the expected output schema and order.

In [0]:
stocks_data = [("Alpha","Buy",1,100),("Alpha","Sell",5,140),("Beta","Buy",2,80),("Beta","Sell",6,60)]
stocks_df = spark.createDataFrame(stocks_data, ["stock_name","operation","operation_day","price"])
display(stocks_df)

output_df = (
    stocks_df
    .withColumn(
        "gain_loss",
        when(col("operation") == "Buy", -col("price"))
        .otherwise(col("price"))
    )
    .groupBy("stock_name")
    .agg(
        sum("gain_loss").alias("capital_gain_loss")
    )
    .select("stock_name", "capital_gain_loss")
    .orderBy("stock_name")
)

display(output_df)


stock_name,operation,operation_day,price
Alpha,Buy,1,100
Alpha,Sell,5,140
Beta,Buy,2,80
Beta,Sell,6,60


stock_name,capital_gain_loss
Alpha,40
Beta,-20


## Que16: Host Popularity Based Rental Pricing

**Difficulty:** Medium

### Problem

You run a short-term rental marketplace and want to see how pricing varies with a host's track record. Each row in `host_searches` is one listing returned in search, with its nightly price and the host's total `number_of_reviews`. Assign every listing a popularity rating from its review count, then report the minimum, average, and maximum nightly price for each rating.

The popularity rating is defined as:
- `New`: 0 reviews
- `Rising`: 1 to 5 reviews
- `Trending Up`: 6 to 15 reviews
- `Popular`: 16 to 40 reviews
- `Hot`: more than 40 reviews

Return one row per rating with `popularity_rating`, `min_price`, `avg_price` (rounded to 2 decimals), and `max_price`.

**Schema columns:** `host_searches.host_id`, `host_searches.number_of_reviews`, `host_searches.price`, `host_searches.room_type`, `host_searches.city`

**Output columns:** `popularity_rating`, `min_price`, `avg_price`, `max_price`

### Examples

#### Example 1

**Input:**

**host_searches:**

| host_id | number_of_reviews | price | room_type | city |
|--------:|------------------:|------:|-----------|------|
| 1001 | 0 | 85 | Entire home | San Francisco |
| 1001 | 0 | 90 | Private room | San Francisco |
| 1002 | 2 | 95 | Entire home | New York |
| 1002 | 3 | 115.51 | Private room | New York |
| 1002 | 5 | 150 | Shared room | New York |
| 1003 | 6 | 85.5 | Entire home | Chicago |
| 1004 | 16 | 130 | Entire home | Boston |
| 1005 | 45 | 360 | Entire home | Miami |

**Output:**

| popularity_rating | min_price | avg_price | max_price |
|-------------------|----------:|----------:|----------:|
| New | 85 | 87.50 | 90 |
| Trending Up | 85.5 | 85.50 | 85.5 |
| Rising | 95 | 120.17 | 150 |
| Popular | 130 | 130.00 | 130 |
| Hot | 360 | 360.00 | 360 |

**Explanation:** The three New York listings (2, 3, and 5 reviews) all fall in the 1-to-5 range (Rising): prices 95, 115.51, and 150 give min 95, max 150, and average 120.17. The single Chicago listing has 6 reviews, so it becomes Trending Up with all three prices equal to 85.5.

### Constraints

- Review-count ranges are inclusive at both ends (e.g., 5 reviews is Rising and 6 reviews is Trending Up).
- Round the average price to 2 decimal places.
- A rating appears only if at least one listing falls in it.
- Order the result by `min_price` in ascending order.
- Return results matching the expected output schema and order.

In [0]:
host_searches_data = [(1001,0,85.0,"Entire home","San Francisco"),(1001,0,90.0,"Private room","San Francisco"),(1002,2,95.0,"Entire home","New York"),(1002,3,115.51,"Private room","New York"),(1002,5,150.0,"Shared room","New York"),(1003,6,85.5,"Entire home","Chicago"),(1004,16,130.0,"Entire home","Boston"),(1005,45,360.0,"Entire home","Miami")]
host_searches_df = spark.createDataFrame(host_searches_data, ["host_id","number_of_reviews","price","room_type","city"])

host_searches_df = (
host_searches_df
    .withColumn("popularity_rating", 
        when(col("number_of_reviews") == 0, "New")
        .when(col("number_of_reviews") <= 5, "Rising")
        .when(col("number_of_reviews") <= 15, "Trending Up")
        .when(col("number_of_reviews") <= 40, "Popular")
        .otherwise("Hot")
    )
)

# 	min_price	avg_price	max_price
output_df = (
host_searches_df.groupBy("popularity_rating").agg(
    min("price").alias("min_price"),
    round(avg("price"), 2).alias("avg_price"),
    max("price").alias("max_price")
)
.orderBy("min_price")
)


display(output_df)



popularity_rating,min_price,avg_price,max_price
New,85.0,87.5,90.0
Trending Up,85.5,85.5,85.5
Rising,95.0,120.17,150.0
Popular,130.0,130.0,130.0
Hot,360.0,360.0,360.0


## Que17: Delivery Performance and Driver Reliability Scoring

**Difficulty:** Hard

### Problem

DoorDash wants to score how reliably each driver delivers. Every delivery is classified by how its actual delivery time compares to the estimated time, and those classifications are combined into a single reliability score.

For each driver, report:
- `total_deliveries`: the number of deliveries the driver completed.
- `on_time_pct`: the percentage of the driver's deliveries whose `actual_time` is at or below the `estimated_time`.
- `late_pct`: the percentage whose `actual_time` is above `estimated_time` but no more than `estimated_time + 5` minutes.
- `very_late_pct`: the percentage whose `actual_time` is more than `estimated_time + 5` minutes.
- `reliability_score`: `(on_time_pct / 100) * 0.5 + (1 - very_late_pct / 100) * 0.3 + min(total_deliveries / 50, 1) * 0.2`.

Report one row per driver, ordered by `driver_id`.

**Schema columns:** `deliveries.delivery_id`, `deliveries.driver_id`, `deliveries.estimated_time`, `deliveries.actual_time`, `deliveries.order_value`

**Output columns:** `driver_id`, `total_deliveries`, `on_time_pct`, `late_pct`, `very_late_pct`, `reliability_score`

### Examples

#### Example 1

**Input:**

**deliveries:**

| delivery_id | driver_id | estimated_time | actual_time | order_value |
|------------:|-----------|---------------:|------------:|------------:|
| 1 | D001 | 30 | 28 | 45.5 |
| 6 | D001 | 25 | 25 | 38.75 |
| 3 | D001 | 35 | 37 | 58.25 |
| 5 | D002 | 30 | 35 | 42.5 |
| 13 | D002 | 35 | 42 | 52.25 |
| 7 | D004 | 40 | 50 | 75 |

**Output:**

| driver_id | total_deliveries | on_time_pct | late_pct | very_late_pct | reliability_score |
|-----------|----------------:|------------:|---------:|-------------:|------------------:|
| D001 | 3 | 66.67 | 33.33 | 0.00 | 0.65 |
| D002 | 2 | 0.00 | 50.00 | 50.00 | 0.16 |
| D004 | 1 | 0.00 | 0.00 | 100.00 | 0.00 |

**Explanation:** D001 made 3 deliveries: two arrived on time and one was late (37 vs 35, 2 minutes over), so `on_time_pct` is 66.67 and `late_pct` is 33.33 with no very-late deliveries.

### Constraints

- Percentages and `reliability_score` are rounded to 2 decimal places.
- `on_time` when `actual_time <= estimated_time`; `late` when `estimated_time < actual_time <= estimated_time + 5`; `very_late` when `actual_time > estimated_time + 5`.
- The volume term `min(total_deliveries / 50, 1)` is capped at 1.0.
- Order the result by `driver_id` ascending.
- Return results matching the expected output schema and order.

In [0]:
deliveries_data = [(1,"D001",30,28,45.5),(6,"D001",25,25,38.75),(3,"D001",35,37,58.25),(5,"D002",30,35,42.5),(13,"D002",35,42,52.25),(7,"D004",40,50,75.0)]
deliveries_df = spark.createDataFrame(deliveries_data, ["delivery_id","driver_id","estimated_time","actual_time","order_value"])

deliveries_df = (
deliveries_df
    .withColumn("on_time", when(col("actual_time") <= col("estimated_time"), 1).otherwise(0))
    .withColumn("late", when((col("actual_time") > col("estimated_time")) & (col("actual_time") <= col("estimated_time") + 5), 1).otherwise(0))
    .withColumn("very_late", when((col("actual_time") > col("estimated_time")) & (col("actual_time") >= col("estimated_time") + 5), 1).otherwise(0))
)

display(deliveries_df)

# driver_id	total_deliveries	on_time_pct	late_pct	very_late_pct	reliability_score

grouped_df = (
deliveries_df.groupBy("driver_id").agg(
    count(col("delivery_id")).alias("total_deliveries"),
    round(((sum(col("on_time"))) / (count(col("delivery_id")))) * 100, 2).alias("on_time_pct"),
    round(((sum(col("late"))) / (count(col("delivery_id")))) * 100, 2).alias("late_pct"),
    round(((sum(col("very_late"))) / (count(col("delivery_id")))) * 100, 2).alias("very_late_pct"),
))

# (on_time_pct / 100) * 0.5 + (1 - very_late_pct / 100) * 0.3 + min(total_deliveries / 50, 1) * 0.2

output_df = (
    grouped_df
    .withColumn(
        "reliability_score",
        round(
            (col("on_time_pct") / 100) * 0.5
            + (1 - col("very_late_pct") / 100) * 0.3
            + least(
                col("total_deliveries") / 50,
                lit(1)
            ) * 0.2,
            2
        )
    )
)

display(output_df)


delivery_id,driver_id,estimated_time,actual_time,order_value,on_time,late,very_late
1,D001,30,28,45.5,1,0,0
6,D001,25,25,38.75,1,0,0
3,D001,35,37,58.25,0,1,0
5,D002,30,35,42.5,0,1,1
13,D002,35,42,52.25,0,0,1
7,D004,40,50,75.0,0,0,1


driver_id,total_deliveries,on_time_pct,late_pct,very_late_pct,reliability_score
D001,3,66.67,33.33,0.0,0.65
D002,2,0.0,50.0,100.0,0.01
D004,1,0.0,0.0,100.0,0.0


## Que18: Sentiment Analysis on Text

**Difficulty:** Hard

### Problem

A retailer scores each review by counting specified positive keyword occurrences and subtracting specified negative keyword occurrences, case-insensitively. For each product, report the review count, average sentiment score, number of reviews with a positive score, and number with a negative score. A zero-score review is neutral.

**Schema columns:** `product_reviews.review_id`, `product_reviews.product_id`, `product_reviews.review_text`, `product_reviews.review_date`

**Output columns:** `product_id`, `total_reviews`, `avg_sentiment_score`, `positive_review_count`, `negative_review_count`

### Examples

#### Example 1

**Input:**

**product_reviews:**

| review_id | product_id | review_text | review_date |
|----------:|-----------:|-------------|-------------|
| 1 | 101 | This product is great and amazing | 2024-01-15 |
| 2 | 101 | Terrible quality worst experience ever | 2024-01-16 |
| 3 | 101 | Love it fantastic product | 2024-01-17 |

**Output:**

| product_id | total_reviews | avg_sentiment_score | positive_review_count | negative_review_count |
|-----------:|-------------:|--------------------:|----------------------:|----------------------:|
| 101 | 3 | 0.67 | 2 | 1 |

**Explanation:** The three review scores are 2, -2, and 2, which average to 0.67.

### Constraints

- Positive keywords: `great`, `good`, `excellent`, `amazing`, `love`, `best`, `fantastic`, `wonderful`.
- Negative keywords: `bad`, `terrible`, `awful`, `worst`, `poor`, `hate`, `horrible`, `disappointing`.
- Count keyword occurrences case-insensitively and round the average score to 2 decimal places.
- Order results by `product_id` ascending.
- Return results matching the expected output schema and order.

In [0]:
product_reviews_data = [(1,101,"This product is great and amazing","2024-01-15"),(2,101,"Terrible quality worst experience ever","2024-01-16"),(3,101,"Love it fantastic product","2024-01-17")]
product_reviews_df = spark.createDataFrame(product_reviews_data, ["review_id","product_id","review_text","review_date"])

product_reviews_df = product_reviews_df.withColumn("review_text", lower(col("review_text")))

from functools import reduce

positive_keywords = ["great", "good", "excellent", "amazing", "love", "best", "fantastic", "wonderful"]
negative_keywords = ["bad", "terrible", "awful", "worst", "poor", "hate", "horrible", "disappointing"]

positive_score = reduce(
    lambda x, y: x + y,
    [when(col("review_text").contains(word), 1).otherwise(0) for word in positive_keywords]
)

negative_score = reduce(
    lambda x, y: x + y,
    [when(col("review_text").contains(word), 1).otherwise(0) for word in negative_keywords]
)

scored_df = (
product_reviews_df
    .withColumn("positive_score", positive_score)
    .withColumn("negative_score", negative_score)
    .withColumn("sentiment_score", col("positive_score") - col("negative_score"))
)

grouped_df = (
scored_df.groupBy("product_id").agg(
    count("review_id").alias("taotal_reviews"),
    round(avg("sentiment_score"),2).alias("avg_sentiment_score"),
    count(when(col("positive_score") > 1, 1)).alias("positive_review_count"),
    count(when(col("negative_score") > 1, 1)).alias("negative_review_count"),

)
.orderBy("product_id")
)


display(grouped_df)





product_id,taotal_reviews,avg_sentiment_score,positive_review_count,negative_review_count
101,3,0.67,2,1


## Que19: Data Freshness SLA Monitoring

**Difficulty:** Hard

### Problem

Monitor data pipeline freshness against SLA thresholds, flagging tables that have exceeded their maximum acceptable delay.

Using the fixed reference time `2024-06-15 12:00:00`, compute for every table in `pipeline_logs` how many hours have passed since its `last_updated` timestamp (`delay_hours`, rounded to 1 decimal place). Join `sla_config` on `table_name` to get the SLA limit and priority, and set `is_breach` to 1 when `delay_hours` exceeds `max_delay_hours`, else 0. `last_updated` must be returned as a string formatted `YYYY-MM-DD HH24:MI:SS`.

**Schema columns:** `pipeline_logs.table_name`, `pipeline_logs.last_updated`, `pipeline_logs.expected_update_frequency_hours`, `sla_config.table_name`, `sla_config.max_delay_hours`, `sla_config.priority`

**Output columns:** `table_name`, `last_updated`, `delay_hours`, `max_delay_hours`, `is_breach`, `priority`

Order the result by `is_breach` descending, then `delay_hours` descending, then `table_name` ascending.

### Examples

#### Example 1

**Input:**

**pipeline_logs:**

| table_name | last_updated | expected_update_frequency_hours |
|------------|---------------------|--------------------------------:|
| users | 2024-06-15 11:30:00 | 1 |
| products | 2024-06-15 10:00:00 | 2 |
| orders | 2024-06-15 09:15:00 | 4 |
| customers | 2024-06-15 06:00:00 | 8 |
| transactions | 2024-06-15 10:30:00 | 1 |

**sla_config:**

| table_name | max_delay_hours | priority |
|------------|----------------:|----------|
| users | 2 | high |
| products | 4 | high |
| orders | 6 | medium |
| customers | 12 | medium |
| transactions | 2 | high |

**Output:**

| table_name | last_updated | delay_hours | max_delay_hours | is_breach | priority |
|------------|---------------------|------------:|----------------:|----------:|----------|
| customers | 2024-06-15 06:00:00 | 6.0 | 12 | 0 | medium |
| orders | 2024-06-15 09:15:00 | 2.8 | 6 | 0 | medium |
| products | 2024-06-15 10:00:00 | 2.0 | 4 | 0 | high |
| transactions | 2024-06-15 10:30:00 | 1.5 | 2 | 0 | high |
| users | 2024-06-15 11:30:00 | 0.5 | 2 | 0 | high |

### Constraints

- Reference time: `2024-06-15 12:00:00` (fixed).
- `delay_hours` = hours between `last_updated` and the reference time, rounded to 1 decimal place.
- `is_breach` = 1 when `delay_hours > max_delay_hours`, else 0.
- `last_updated` must be returned as a string formatted `YYYY-MM-DD HH24:MI:SS`.
- Priority values: `high`, `medium`, `low`.
- Order the result by `is_breach` descending, then `delay_hours` descending, then `table_name` ascending.
- Return results matching the expected output schema and order.

In [0]:
pipeline_logs_data = [("users","2024-06-15 11:30:00",1),("products","2024-06-15 10:00:00",2),("orders","2024-06-15 09:15:00",4),("customers","2024-06-15 06:00:00",8),("transactions","2024-06-15 10:30:00",1)]
pipeline_logs_df = spark.createDataFrame(pipeline_logs_data, ["table_name","last_updated","expected_update_frequency_hours"])

pipeline_logs_df = pipeline_logs_df.withColumn("last_updated", date_format(col("last_updated"), "yyyy-MM-dd HH:mm:ss"))

sla_config_data = [("users",2,"high"),("products",4,"high"),("orders",6,"medium"),("customers",12,"medium"),("transactions",2,"high")]
sla_config_df = spark.createDataFrame(sla_config_data, ["table_name","max_delay_hours","priority"])

display(pipeline_logs_df)
display(sla_config_df)

refrenced_time = date_format(lit("2024-06-15 12:00:00"), "yyyy-MM-dd HH:mm:ss")


joined_df = (
pipeline_logs_df.alias("p").join(sla_config_df.alias("s"), on="table_name", how="inner")
.select("table_name", "last_updated", "max_delay_hours", "priority")
)

output_df = (
joined_df
    .withColumn("delay_hours", round((unix_timestamp(refrenced_time) - unix_timestamp(col("last_updated"))) / 3600, 1))
    .withColumn("is_breach", when(col("max_delay_hours") < col("delay_hours"), 1).otherwise(0))
).orderBy(col("is_breach").desc(), col("delay_hours").desc(), col("table_name"))


display(output_df)


table_name,last_updated,expected_update_frequency_hours
users,2024-06-15 11:30:00,1
products,2024-06-15 10:00:00,2
orders,2024-06-15 09:15:00,4
customers,2024-06-15 06:00:00,8
transactions,2024-06-15 10:30:00,1


table_name,max_delay_hours,priority
users,2,high
products,4,high
orders,6,medium
customers,12,medium
transactions,2,high


table_name,last_updated,max_delay_hours,priority,delay_hours,is_breach
customers,2024-06-15 06:00:00,12,medium,6.0,0
orders,2024-06-15 09:15:00,6,medium,2.8,0
products,2024-06-15 10:00:00,4,high,2.0,0
transactions,2024-06-15 10:30:00,2,high,1.5,0
users,2024-06-15 11:30:00,2,high,0.5,0


## Que20: Revenue Anomaly Detection (±200% of Moving Avg)

**Difficulty:** Hard

### Problem

A revenue-monitoring team compares each product-day with recent history. For each row, compute `moving_avg_7d` as the average revenue of up to the six previous recorded rows for that product, excluding the current row, and mark `is_anomaly` as `Yes` when the current revenue differs from that baseline by more than 200%.

**Schema columns:** `daily_revenue.revenue_date`, `daily_revenue.product_id`, `daily_revenue.revenue`

**Output columns:** `revenue_date`, `product_id`, `revenue`, `moving_avg_7d`, `is_anomaly`

### Examples

#### Example 1

**Input:**

**daily_revenue:**

| revenue_date | product_id | revenue |
|-------------|----------:|--------:|
| 2024-01-01 | 101 | 1000 |
| 2024-01-02 | 101 | 1100 |
| 2024-01-03 | 101 | 950 |
| 2024-01-04 | 101 | 1050 |
| 2024-01-05 | 101 | 1000 |

**Output:**

| revenue_date | product_id | revenue | moving_avg_7d | is_anomaly |
|-------------|----------:|--------:|--------------:|-----------|
| 2024-01-01 | 101 | 1000 | NULL | No |
| 2024-01-02 | 101 | 1100 | 1000.00 | No |
| 2024-01-03 | 101 | 950 | 1050.00 | No |
| 2024-01-04 | 101 | 1050 | 1016.67 | No |
| 2024-01-05 | 101 | 1000 | 1025.00 | No |

**Explanation:** Each baseline uses only earlier rows, and none of the shown revenues differs from it by more than 200%.

### Constraints

- Use up to six previous recorded rows for the same product and exclude the current row.
- Leave `moving_avg_7d` null on a product's first row and mark that row `No`.
- Mark `Yes` when `abs(revenue - moving_avg_7d) / moving_avg_7d > 2.0`.
- Round the baseline to two decimals and order by `product_id`, then `revenue_date`.
- Return results matching the expected output schema and order.

In [0]:
daily_revenue_data = [("2024-01-01", 101, 1000),("2024-01-02", 101, 1100),("2024-01-03", 101, 950),("2024-01-04", 101, 1050),("2024-01-05", 101, 1000),("2024-01-06", 101, 1080),("2024-01-07", 101, 1020),("2024-01-08", 101, 1050),("2024-01-09", 101, 5000),("2024-01-10", 101, 980),("2024-01-01", 102, 500),("2024-01-02", 102, 520),("2024-01-03", 102, 510),("2024-01-04", 102, 495),("2024-01-05", 102, 505),("2024-01-06", 102, 515),("2024-01-07", 102, 500),("2024-01-08", 102, 1500),("2024-01-09", 102, 490),("2024-01-10", 102, 505),("2024-01-02", 103, 2000),("2024-01-04", 103, 2100),("2024-01-07", 103, 1950),("2024-01-10", 103, 2050),("2024-01-12", 103, 2025),("2024-01-15", 103, 1980),("2024-01-18", 103, 2010),("2024-01-20", 103, 100),("2024-01-22", 103, 2040)]

daily_revenue_df = spark.createDataFrame(daily_revenue_data,["revenue_date", "product_id", "revenue"])
# display(daily_revenue_df)

window_spec = Window.partitionBy("product_id").orderBy("revenue_date").rowsBetween(-6, -1)

moving_avg_df = (
daily_revenue_df
    .withColumn("moving_avg_7d", round(avg("revenue").over(window_spec), 2))
)

output_df = (
moving_avg_df
    .withColumn("is_anomaly",
    when((abs(col("revenue") - col("moving_avg_7d")) / col("moving_avg_7d")) > 2.0, "Yes").otherwise("No"))
)

display(output_df)



revenue_date,product_id,revenue,moving_avg_7d,is_anomaly
2024-01-01,101,1000,null,No
2024-01-02,101,1100,1000.0,No
2024-01-03,101,950,1050.0,No
2024-01-04,101,1050,1016.67,No
2024-01-05,101,1000,1025.0,No
2024-01-06,101,1080,1020.0,No
2024-01-07,101,1020,1030.0,No
2024-01-08,101,1050,1033.33,No
2024-01-09,101,5000,1025.0,Yes
2024-01-10,101,980,1700.0,No
